In [ ]:
!pip install -q numpy
!pip install -q pandas
!pip install -q librosa
!pip install -q tensorflow
!pip install -q scikit-learn
!pip install -q keras-tuner

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf

from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight

# Data & Audio settings
SR = 22050
N_MELS = 128
FMAX = 8000
FIXED_TIME_STEPS = 150
NUM_CLASSES = 10

# Training parameters
BATCH_SIZE = 32
EPOCHS = 50
N_FOLDS = 5
SEED = 42

# Paths
TRAIN_CSV_PATH = "dataset/Train.csv"
TRAIN_AUDIO_DIR = "dataset/Train"
TEST_CSV_PATH = "dataset/Test_Public.csv"
TEST_AUDIO_DIR = "dataset/Test_Public"
CACHE_DIR = "dataset/mel_cache"  # Directory for caching audio features

# Set random seed
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Create cache directory if needed
os.makedirs(CACHE_DIR, exist_ok=True)

# Load and Inspect Metadata
df = pd.read_csv(TRAIN_CSV_PATH)
df["filepath"] = df["file_name"].apply(lambda x: os.path.join(TRAIN_AUDIO_DIR, x))

# Checking for missing files
missing = [f for f in df["filepath"] if not os.path.exists(f)]
print(f"Missing audio files: {len(missing)}")
if missing:
    print(missing[:5])  # Show some samples


file_paths = df["filepath"].values
labels = df["classID"].values
assert len(file_paths) == len(labels), "Mismatch in paths and labels"


#Making sure all the class labels are correct and there
assert labels.min() == 0 and labels.max() == NUM_CLASSES - 1, "Label range mismatch!"
assert set(np.unique(labels)) == set(range(NUM_CLASSES)), "Missing some classes!"

# Optional: check distribution
for cls, count in pd.Series(labels).value_counts().sort_index().items():
    print(f"Class {cls}: {count} samples")

#  Mel Spectrogram

In [ ]:
def load_and_cache_audio(file_path):
    """Load audio and cache mel spectrogram for faster access"""
    cache_file = os.path.join(
        CACHE_DIR,
        os.path.basename(file_path).replace('.wav', '.npy')
    )

    # If cache exists and loads correctly, use it
    if os.path.exists(cache_file):
        try:
            mel_db = np.load(cache_file)
            #print(f"[SKIP] Using cached: {os.path.basename(cache_file)}")
            return mel_db
        except Exception:
            print(f"[WARN] Failed to load cache: {os.path.basename(cache_file)} — Regenerating.")

    # Generate mel spectrogram
    try:
        y, sr = librosa.load(file_path, sr=SR)
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, fmax=FMAX)
        mel_db = librosa.power_to_db(mel, ref=np.max)

        # Normalize
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)

        # Add channel dimension
        mel_db = np.expand_dims(mel_db, axis=-1)

        # Cache the result
        np.save(cache_file, mel_db)
        #print(f"[NEW] Cached: {os.path.basename(cache_file)}")

        return mel_db
    except Exception as e:
        print(f"[ERROR] Failed to process {file_path}: {e}")
        return np.zeros((N_MELS, FIXED_TIME_STEPS, 1))



Class 0: 800 samples
Class 1: 368 samples
Class 2: 800 samples
Class 3: 800 samples
Class 4: 800 samples
Class 5: 801 samples
Class 6: 291 samples
Class 7: 828 samples
Class 8: 769 samples
Class 9: 800 samples


In [ ]:
from tqdm import tqdm
import os

def cache_all_audio(file_paths):
    print(" Pre-caching mel spectrograms...\n")
    skipped, created, failed = 0, 0, 0

    for path in tqdm(file_paths):
        cache_file = os.path.join(CACHE_DIR, os.path.basename(path).replace('.wav', '.npy'))

        # Check if already cached before calling the function
        already_cached = os.path.exists(cache_file)

        result = load_and_cache_audio(path)

        if result is None or np.count_nonzero(result) == 0:
            failed += 1
        elif already_cached:
            skipped += 1
        else:
            created += 1

    print(f"\n Done caching.")
    print(f" Created: {created}")
    print(f" Skipped: {skipped}")
    print(f" Failed:  {failed}")


cache_all_audio(file_paths)

In [ ]:
import random

def inspect_random_cached_mels(n=4):
    cached_files = [f for f in os.listdir(CACHE_DIR) if f.endswith('.npy')]
    if len(cached_files) == 0:
        print("No cached mel spectrograms found.")
        return

    print(f"\nInspecting {n} random cached mel spectrograms:\n")

    for fname in random.sample(cached_files, min(n, len(cached_files))):
        path = os.path.join(CACHE_DIR, fname)
        try:
            mel = np.load(path)
            print(f" {fname}")
            print(f"   Shape     : {mel.shape}")
            print(f"   Min/Max   : {mel.min():.4f} / {mel.max():.4f}")
            print(f"   Non-zero% : {np.count_nonzero(mel) / mel.size * 100:.2f}%")
            print("---------------")
        except Exception as e:
            print(f" Failed to load {fname}: {e}")

# Run it
inspect_random_cached_mels()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import random

def visualize_random_cached_mels(n=4):
    cached_files = [f for f in os.listdir(CACHE_DIR) if f.endswith('.npy')]
    if len(cached_files) == 0:
        print(" No cached mel spectrograms found.")
        return

    selected_files = random.sample(cached_files, min(n, len(cached_files)))

    plt.figure(figsize=(15, 4))
    for i, fname in enumerate(selected_files):
        mel = np.load(os.path.join(CACHE_DIR, fname))

        # Squeeze last dimension to get (128, time) for plotting
        mel = np.squeeze(mel, axis=-1)

        plt.subplot(1, n, i + 1)
        plt.imshow(mel, origin='lower', aspect='auto', cmap='magma')
        plt.title(fname)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# Run it
visualize_random_cached_mels()


# Dataset  -- Creating Augmented Dataset

In [ ]:
# TensorFlow-optimized augmentation function
@tf.function
def tf_augment_spectrogram(spectrogram):
    orig_shape = tf.shape(spectrogram)
    height = orig_shape[0]
    width = orig_shape[1]

    stretch_factor = tf.random.uniform([], 0.8, 1.2)
    new_width = tf.cast(tf.cast(width, tf.float32) * stretch_factor, tf.int32)
    x = tf.image.resize(spectrogram, [height, new_width])
    x = tf.image.resize(x, [height, width])

    freq_mask_param = tf.cast(tf.cast(height, tf.float32) * 0.15, tf.int32)
    f0 = tf.random.uniform([], 0, height - freq_mask_param, dtype=tf.int32)
    mask = tf.ones((freq_mask_param, width, 1), dtype=tf.float32)
    paddings = [[f0, height - f0 - freq_mask_param], [0, 0], [0, 0]]
    freq_mask = tf.pad(mask, paddings, constant_values=0.0)
    x = x * (1.0 - freq_mask)

    noise = tf.random.normal(shape=tf.shape(x), mean=0.0, stddev=0.01)
    x = x + noise

    x_min = tf.reduce_min(x)
    x_max = tf.reduce_max(x)
    x = (x - x_min) / (x_max - x_min + 1e-6)

    return x


# Improved hybrid dataset creation
def create_hybrid_dataset(file_paths, labels, batch_size=32, augment=True):
    """Create dataset with cached features but on-the-fly augmentation"""
    
    def process_path(file_path, label):
        def _load_and_process(path):
            path = path.numpy().decode("utf-8")  
            mel_db = load_and_cache_audio(path)
        
            if mel_db.shape[0] != N_MELS:
                print(f"[WARN] {path}: Expected {N_MELS} mel bins, got {mel_db.shape[0]}. Skipping.")
                mel_db = np.zeros((N_MELS, FIXED_TIME_STEPS), dtype=np.float32)
        
            if mel_db.shape[1] != FIXED_TIME_STEPS:
                mel_db = librosa.util.fix_length(np.squeeze(mel_db, axis=-1), size=FIXED_TIME_STEPS, axis=1)
                mel_db = np.expand_dims(mel_db, axis=-1)
            elif mel_db.ndim == 2:
                mel_db = np.expand_dims(mel_db, axis=-1)
        
            return mel_db.astype(np.float32)

        
        # 👇 Pass file_path into the py_function call
        mel_db = tf.py_function(
            func=_load_and_process,
            inp=[file_path],
            Tout=tf.float32
        )
    
        mel_db = tf.ensure_shape(mel_db, [N_MELS, FIXED_TIME_STEPS, 1])
        return mel_db, tf.cast(label, tf.int32)


    # Create base dataset
    dataset = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    dataset = dataset.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)

    # Optional: apply augmentation
    if augment:
        dataset = dataset.map(
            lambda x, y: (
                tf.cond(
                    tf.random.uniform([], 0, 1) < 0.7,
                    lambda: tf_augment_spectrogram(x),
                    lambda: x
                ),
                y
            ),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    # Shuffle, batch, prefetch
    dataset = dataset.shuffle(buffer_size=min(1000, len(file_paths)))
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


NameError: name 'tf' is not defined

### Testing the see if it works

In [ ]:
# Grab a small sample to test
test_file_paths = file_paths[:16]
test_labels = labels[:16]

# Create the hybrid dataset (augmentation ON for testing)
test_ds = create_hybrid_dataset(test_file_paths, test_labels, batch_size=4, augment=True)

# Take one batch and inspect
for x_batch, y_batch in test_ds.take(1):
    print("Loaded batch successfully!")
    print("Spectrogram batch shape:", x_batch.shape)  # Should be (4, 128, 150, 1)
    print("Labels shape:", y_batch.shape)              # Should be (4,)
    print("Label values:", y_batch.numpy())


# Model

In [ ]:
# Keep your original model architecture
def build_model(hp):
    lr = hp.Float("learning_rate", 1e-4, 1e-3, sampling="log")
    dropout = hp.Float("dropout", 0.3, 0.6, step=0.1)
    l2_val = hp.Float("l2", 1e-5, 1e-3, sampling="log")
    
    inputs = tf.keras.layers.Input(shape=(128, FIXED_TIME_STEPS, 1))

    # Initial block (unchanged)
    x = layers.Conv2D(32, (3, 3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Residual Block 1
    residual = x
    x = layers.Conv2D(32, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(32, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.add([x, residual])
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Residual Block 2
    residual = layers.Conv2D(64, (1, 1), padding='same')(x)
    x = layers.Conv2D(64, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.add([x, residual])
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Residual Block 3
    residual = layers.Conv2D(128, (1, 1), padding='same')(x)
    x = layers.Conv2D(128, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(128, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.add([x, residual])
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Final Dense Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(l2_val))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)

    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = tf.keras.models.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# Tuning Hyperparameter 

In [ ]:
# ---- Updated training code with hybrid dataset ----

# For hyperparameter tuning
import keras_tuner as kt
from sklearn.model_selection import train_test_split

def run_hyperparameter_tuning():
    """Run hyperparameter tuning with hybrid dataset approach"""
    TUNER_DIR = "vol2_tuner_results"
    
    # Create train/val split for tuner
    tune_train_paths, tune_val_paths, tune_train_labels, tune_val_labels = train_test_split(
        file_paths,
        labels,
        test_size=0.2,
        stratify=labels,
        random_state=SEED
    )
    
    # Create datasets with hybrid approach
    tune_train_ds = create_hybrid_dataset(tune_train_paths, tune_train_labels, BATCH_SIZE, augment=True)
    tune_val_ds = create_hybrid_dataset(tune_val_paths, tune_val_labels, BATCH_SIZE, augment=False)
    
    # Define tuner
    tuner = kt.Hyperband(
        build_model,
        objective='val_accuracy',
        max_epochs=13,
        factor=3,
        hyperband_iterations=1,
        directory=TUNER_DIR,
        project_name='mel_cnn_finetune',
        overwrite=False
    )
    
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True
    )
    
    # Run tuner
    tuner.search(
        tune_train_ds,
        validation_data=tune_val_ds,
        epochs=15,
        callbacks=[early_stopping],
        verbose=1
    )
    
    # Get best hyperparameters
    best_hp = tuner.get_best_hyperparameters(1)[0]
    print("\nBest Hyperparameters Found:")
    for param in best_hp.values:
        print(f"{param}: {best_hp.get(param)}")
        
    return best_hp

Trial 30 Complete [00h 54m 52s]
val_accuracy: 0.7471671104431152

Best val_accuracy So Far: 0.847733736038208
Total elapsed time: 17h 10m 53s

Best Hyperparameters Found:
learning_rate: 0.0003725300187883502
dropout: 0.5
l2: 0.0004489080930912622
tuner/epochs: 13
tuner/initial_epoch: 5
tuner/bracket: 2
tuner/round: 2
tuner/trial_id: 0015


# Train  

In [ ]:
# Updated K-Fold training with hybrid dataset
def train_kfold(best_hp):
    """Train with k-fold cross-validation using hybrid datasets"""
    MODEL_DIR = os.path.join(os.getcwd(), "model_v2")
    os.makedirs(MODEL_DIR, exist_ok=True)
    
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_results = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(file_paths, labels)):
        print(f"\n Starting Fold {fold + 1}/{N_FOLDS}")
        
        # Split data for this fold
        train_files = file_paths[train_idx]
        val_files = file_paths[val_idx]
        train_labels_fold = labels[train_idx]
        val_labels_fold = labels[val_idx]
        
        # Compute class weights
        class_weights = compute_class_weight('balanced', classes=np.unique(train_labels_fold), y=train_labels_fold)
        class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}
        
        # Create hybrid datasets
        train_ds = create_hybrid_dataset(train_files, train_labels_fold, BATCH_SIZE, augment=True)
        val_ds = create_hybrid_dataset(val_files, val_labels_fold, BATCH_SIZE, augment=False)
        
        # Build model with best hyperparams
        model = build_model(best_hp)
        
        # Define callbacks
        model_path = f"model_v2/fold_{fold + 1}_best_model.h5"
        callbacks = [
            EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_accuracy', patience=5, factor=0.5, min_lr=1e-5, verbose=1),
            ModelCheckpoint(model_path, monitor='val_accuracy', save_best_only=True, verbose=1)
        ]
        
        # Train
        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=EPOCHS,
            class_weight=class_weight_dict,
            callbacks=callbacks,
            verbose=1
        )
        
        # Record best fold metrics
        best_val_acc = max(history.history['val_accuracy'])
        best_val_loss = min(history.history['val_loss'])
        
        print(f" Fold {fold + 1}: Best Val Accuracy = {best_val_acc:.4f}, Loss = {best_val_loss:.4f}")
        fold_results.append({
            "fold": fold + 1,
            "val_accuracy": best_val_acc,
            "val_loss": best_val_loss,
            "model_path": model_path
        })
    
    return fold_results
# Example usage:
# best_hp = run_hyperparameter_tuning()
# fold_results = train_kfold(best_hp)


 Starting Fold 1/5
Epoch 1/50
 15/177 [=>............................] - ETA: 8:50 - loss: 2.9596 - accuracy: 0.1229

c:\Users\emink\AppData\Local\Programs\Python\Python311\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1103
  warnings.warn(
c:\Users\emink\AppData\Local\Programs\Python\Python311\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1323
  warnings.warn(
c:\Users\emink\AppData\Local\Programs\Python\Python311\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1523
  warnings.warn(


177/177 [==============================] - ETA: 0s - loss: 2.4546 - accuracy: 0.2009
Epoch 1: val_accuracy improved from -inf to 0.04178, saving model to model\fold_1_best_model.h5
177/177 [==============================] - 620s 3s/step - loss: 2.4546 - accuracy: 0.2009 - val_loss: 4.3641 - val_accuracy: 0.0418 - lr: 3.7253e-04
Epoch 2/50


c:\Users\emink\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\engine\training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


177/177 [==============================] - ETA: 0s - loss: 2.0817 - accuracy: 0.2914
Epoch 2: val_accuracy improved from 0.04178 to 0.04816, saving model to model\fold_1_best_model.h5
177/177 [==============================] - 659s 4s/step - loss: 2.0817 - accuracy: 0.2914 - val_loss: 6.7619 - val_accuracy: 0.0482 - lr: 3.7253e-04
Epoch 3/50
177/177 [==============================] - ETA: 0s - loss: 1.8188 - accuracy: 0.3725
Epoch 3: val_accuracy improved from 0.04816 to 0.19476, saving model to model\fold_1_best_model.h5
177/177 [==============================] - 731s 4s/step - loss: 1.8188 - accuracy: 0.3725 - val_loss: 3.6580 - val_accuracy: 0.1948 - lr: 3.7253e-04
Epoch 4/50
177/177 [==============================] - ETA: 0s - loss: 1.7087 - accuracy: 0.4106
Epoch 4: val_accuracy improved from 0.19476 to 0.43697, saving model to model\fold_1_best_model.h5
177/177 [==============================] - 781s 4s/step - loss: 1.7087 - accuracy: 0.4106 - val_loss: 1.7526 - val_accuracy: 0.4

In [ ]:
# Hyperparameter tuning
best_hp = run_hyperparameter_tuning()

In [ ]:
# Training
fold_results = train_kfold(best_hp)

### Test


In [ ]:
from sklearn.metrics import classification_report, accuracy_score
import tensorflow as tf
import numpy as np
import pandas as pd
import os

# Separate test cache
TEST_CACHE_DIR = "dataset/test_mel_cache"

# Reuse everything else but override cache dir
def load_and_cache_audio_test(file_path):
    """Test version of loader with separate cache"""
    cache_file = os.path.join(
        TEST_CACHE_DIR,
        os.path.basename(file_path).replace('.wav', '.npy')
    )
    
    if os.path.exists(cache_file):
        try:
            return np.load(cache_file)
        except:
            pass

    try:
        y, sr = librosa.load(file_path, sr=SR)
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, fmax=FMAX)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
        mel_db = np.expand_dims(mel_db, axis=-1)
        np.save(cache_file, mel_db)
        return mel_db
    except:
        return np.zeros((N_MELS, FIXED_TIME_STEPS, 1))


# Redefine dataset function with override
def create_test_dataset(file_paths, labels, batch_size=32):
    def process_path(file_path, label):
        def _load(path):
            path = path.numpy().decode("utf-8")
            mel = load_and_cache_audio_test(path)
            if mel.shape[1] != FIXED_TIME_STEPS:
                mel = librosa.util.fix_length(np.squeeze(mel, axis=-1), size=FIXED_TIME_STEPS, axis=1)
                mel = np.expand_dims(mel, axis=-1)
            elif mel.ndim == 2:
                mel = np.expand_dims(mel, axis=-1)
            return mel.astype(np.float32)

        mel = tf.py_function(_load, inp=[file_path], Tout=tf.float32)
        mel = tf.ensure_shape(mel, [N_MELS, FIXED_TIME_STEPS, 1])
        return mel, tf.cast(label, tf.int32)

    ds = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    ds = ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


# Load test set
test_df = pd.read_csv(TEST_CSV_PATH)
test_df["filepath"] = test_df["file_name"].apply(lambda x: os.path.join(TEST_AUDIO_DIR, x))
test_paths = test_df["filepath"].values
test_labels = test_df["classID"].values

# Make test cache directory
os.makedirs(TEST_CACHE_DIR, exist_ok=True)

# Create test dataset
test_ds = create_test_dataset(test_paths, test_labels)

# Load models and evaluate
model_preds = []
accuracies = []
model_paths = sorted([f for f in os.listdir("model_v2") if f.endswith((".h5", ".keras"))])

print(" Evaluating each fold model...\n")

for path in model_paths:
    print(f"→ Loading {path}")
    model = tf.keras.models.load_model(os.path.join("/kaggle/working/model", path))
    
    preds = model.predict(test_ds, verbose=0)
    pred_labels = np.argmax(preds, axis=1)
    
    acc = accuracy_score(test_labels, pred_labels)
    accuracies.append(acc)
    model_preds.append(pred_labels)

    print(f"   Fold Accuracy: {acc:.4f}")

# Stack predictions from all models
model_preds = np.stack(model_preds)  # shape: (num_models, num_samples)

# Ensemble (majority vote)
ensemble_preds = np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=0, arr=model_preds)
ensemble_acc = accuracy_score(test_labels, ensemble_preds)

# Report
print("\n Ensemble Classification Report:")
print(classification_report(test_labels, ensemble_preds, digits=4))
print(f"\n Ensemble Accuracy: {ensemble_acc:.4f}")

# Best fold
best_idx = np.argmax(accuracies)
print("\n Best Individual Fold Model:")
print(f"   Path    : {model_paths[best_idx]}")
print(f"   Accuracy: {accuracies[best_idx]:.4f}")

# Compare
if ensemble_acc > accuracies[best_idx]:
    print("\n Ensemble outperformed the best individual fold.")
else:
    print("\n Best individual fold outperformed the ensemble.")

 Evaluating saved models on Test_Public...

 Fold 1 Evaluation:
 Accuracy: 0.7683
 Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.2000    0.3333        10
           1     1.0000    1.0000    1.0000         3
           2     0.9091    1.0000    0.9524        10
           3     1.0000    0.8000    0.8889        10
           4     0.5333    0.8000    0.6400        10
           5     0.5000    0.3333    0.4000         9
           6     1.0000    1.0000    1.0000         4
           7     0.5714    1.0000    0.7273         8
           8     1.0000    0.8750    0.9333         8
           9     0.8333    1.0000    0.9091        10

    accuracy                         0.7683        82
   macro avg     0.8347    0.8008    0.7784        82
weighted avg     0.8150    0.7683    0.7454        82

--------------------------------------------------
 Fold 2 Evaluation:
 Accuracy: 0.7439
 Classification Report:
              precisio

### Test With Google Drive

In [ ]:
"""
Tarık Buğra Ay - 042101100
"""

import os
import numpy as np
import pandas as pd
import tensorflow as tf
import librosa
import gdown
from tqdm import tqdm

# ===================== CONFIG =====================
MODEL_URL = "https://drive.google.com/uc?export=download&id=1bbGUu1q4-Q8lGD3axKNtrDx1I6qxxras"   #Google Drive
TEST_CSV_PATH = "/kaggle/input/yldz-teknik-proje-1-train-dataset/Test_Public.csv"                #Labels
TEST_AUDIO_DIR = "/kaggle/input/yldz-teknik-proje-1-train-dataset/Test_Public/"                  #Audios

SAMPLE_RATE = 22050
N_MELS = 128
FMAX = 8000
FIXED_TIME_STEPS = 150
BATCH_SIZE = 32
# ===================================================

# Download model from Google Drive
print("Downloading model...")
model_path = gdown.download(MODEL_URL, quiet=False)
model = tf.keras.models.load_model(model_path)
print("Model loaded.\n")

# Load test CSV
df = pd.read_csv(TEST_CSV_PATH)
file_paths = [os.path.join(TEST_AUDIO_DIR, fname) for fname in df["file_name"]]
true_labels = df["classID"].values

# Audio preprocessing function
def load_and_process(file_path):
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE)
    n_fft = min(2048, len(y) // 2)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, fmax=FMAX, n_fft=n_fft)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min())
    mel_db = tf.image.resize(mel_db[..., np.newaxis], (128, FIXED_TIME_STEPS))
    return mel_db.numpy()

# Create dataset
print("Preprocessing test set...")
processed_data = []
for path in tqdm(file_paths):
    mel = load_and_process(path)
    processed_data.append(mel)

X_test = np.stack(processed_data, axis=0)
y_test = np.array(true_labels)

# Predict in batch
print("\nPredicting...")
pred_probs = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)
pred_classes = np.argmax(pred_probs, axis=1)

# Accuracy
accuracy = np.mean(pred_classes == y_test) * 100
print(f"\nAccuracy: {accuracy:.2f}%")

#======================================= NOT============================================================================
# Hocam, bu kısmın sonuçlarını benimle paylaşırsanız çok sevinirim. 
# Böylece modelin nerelerde zayıf kaldığını görüp, diğer modeli daha verimli şekilde geliştirebilirim. Teşekkür ederim.
#=======================================================================================================================

# === Classification Report ===
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


report = classification_report(y_test, pred_classes, output_dict=True)
report_df = pd.DataFrame(report).transpose()
print("\nClassification Report:")
print(report_df)
